In [ ]:
import csv
import subprocess
from pathlib import Path
from typing import Optional, Tuple, List, Dict

# --- CONFIG ---
ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Clone")
OUTDIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5")

WORKFLOW_PATHS = [
    ".github/workflows",       # modern YAML workflows
    ".github/main.workflow",   # legacy HCL workflow format
]
YAML_EXTS = {".yml", ".yaml"}


def run_git(repo_dir: Path, args: List[str]) -> subprocess.CompletedProcess:
    return subprocess.run(
        ["git", "-C", str(repo_dir), *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
    )


def is_git_repo(repo_dir: Path) -> bool:
    r = run_git(repo_dir, ["rev-parse", "--is-inside-work-tree"])
    return r.returncode == 0 and r.stdout.strip() == "true"


def head_has_workflows(repo_dir: Path) -> bool:
    wf_dir = repo_dir / ".github" / "workflows"
    if wf_dir.is_dir():
        # strict: only yml/yaml
        for p in wf_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in YAML_EXTS:
                return True

        # lenient fallback: any file inside workflows dir
        for p in wf_dir.rglob("*"):
            if p.is_file():
                return True

    legacy = repo_dir / ".github" / "main.workflow"
    return legacy.is_file()


def first_last_actions_commits(repo_dir: Path) -> Tuple[Optional[str], Optional[str]]:
    # last commit touching workflow paths
    last = run_git(repo_dir, ["rev-list", "--all", "-n", "1", "--", *WORKFLOW_PATHS])
    last_commit = last.stdout.strip() if last.returncode == 0 else ""
    if not last_commit:
        return None, None

    # first commit touching workflow paths
    first = run_git(repo_dir, ["rev-list", "--all", "--reverse", "-n", "1", "--", *WORKFLOW_PATHS])
    first_commit = first.stdout.strip() if first.returncode == 0 else ""
    if not first_commit:
        first_commit = last_commit

    return first_commit, last_commit


def commit_iso_date(repo_dir: Path, commit: str) -> Optional[str]:
    if not commit:
        return None
    r = run_git(repo_dir, ["show", "-s", "--format=%cI", commit])
    if r.returncode != 0:
        return None
    d = r.stdout.strip()
    return d if d else None


def find_repos(root: Path) -> List[Path]:
    # Assumes each immediate subfolder is a repo (common for cloned datasets)
    return [p for p in root.iterdir() if p.is_dir()]


def scan_repos(root: Path) -> List[Dict]:
    repos = find_repos(root)
    rows: List[Dict] = []

    for repo_dir in repos:
        name = repo_dir.name

        if not is_git_repo(repo_dir):
            rows.append({
                "repo_name": name,
                "repo_path": str(repo_dir),
                "is_git_repo": False,
                "head_has_workflows": False,
                "has_actions_in_history": False,
                "first_actions_commit": "",
                "first_actions_date": "",
                "last_actions_commit": "",
                "last_actions_date": "",
            })
            continue

        head_has = head_has_workflows(repo_dir)
        first_c, last_c = first_last_actions_commits(repo_dir)
        has_hist = first_c is not None

        first_d = commit_iso_date(repo_dir, first_c) if first_c else None
        last_d = commit_iso_date(repo_dir, last_c) if last_c else None

        rows.append({
            "repo_name": name,
            "repo_path": str(repo_dir),
            "is_git_repo": True,
            "head_has_workflows": head_has,
            "has_actions_in_history": has_hist,
            "first_actions_commit": first_c or "",
            "first_actions_date": first_d or "",
            "last_actions_commit": last_c or "",
            "last_actions_date": last_d or "",
        })

    return rows


def write_reports(rows: List[Dict], outdir: Path) -> None:
    outdir.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        "repo_name",
        "repo_path",
        "is_git_repo",
        "head_has_workflows",
        "has_actions_in_history",
        "first_actions_commit",
        "first_actions_date",
        "last_actions_commit",
        "last_actions_date",
    ]

    all_csv = outdir / "all_repos_actions_scan.csv"
    gha_csv = outdir / "gha_repos_history_only.csv"
    gha_txt = outdir / "gha_repos_history_only.txt"

    with all_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    gha_rows = [r for r in rows if r["has_actions_in_history"]]

    with gha_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(gha_rows)

    with gha_txt.open("w", encoding="utf-8") as f:
        for r in gha_rows:
            f.write(r["repo_path"] + "\n")

    total = len(rows)
    git_repos = sum(1 for r in rows if r["is_git_repo"])
    gha_hist = sum(1 for r in rows if r["has_actions_in_history"])
    gha_head = sum(1 for r in rows if r["head_has_workflows"])

    print("Done.")
    print(f"Total folders scanned:      {total}")
    print(f"Git repos found:            {git_repos}")
    print(f"Has Actions in history:     {gha_hist}")
    print(f"Has Actions at HEAD:        {gha_head}")
    print(f"Reports written to:         {outdir.resolve()}")
    print(f" - {all_csv.name}")
    print(f" - {gha_csv.name}")
    print(f" - {gha_txt.name}")
